# Ad Click Prediction
A predictive system that determines the likelihood of a user clicking on an advertisement based on:

Phase 1: Data Exploration & Understanding
--

Step 1: Initial Data Loading

In [3]:
# Read the messy training data CSV file and display the first fe_dfw rows of the DataFrame.
from pathlib import Path
import pandas as pd

csv_path = Path('/Data/Ad_click_prediction_train.csv')
if not csv_path.exists():
    csv_path = Path('Data/Ad_click_prediction_train.csv')

df = pd.read_csv(csv_path)

print("Sales Campaign Dataset Overview:")
print("=" * 80)
display(df.head())

print("Dataset dimensions:", df.shape)
print("Column types:\n", df.dtypes)
print("Target variable (CTR) distribution:\n", df['is_click'].value_counts())
print("Missing values:\n", df.isnull().sum())

Sales Campaign Dataset Overview:


,session_id,DateTime,user_id,product,campaign_id,webpage_id,product_category_1,product_category_2,user_group_id,gender,age_level,user_depth,city_development_index,var_1,is_click
0,140690,2017-07-02 00:00,858557,C,359520,13787,4,NaN,10.0,Female,4.0,3.0,3.0,0,0
1,333291,2017-07-02 00:00,243253,C,105960,11085,5,NaN,8.0,Female,2.0,2.0,NaN,0,0
2,129781,2017-07-02 00:00,243253,C,359520,13787,4,NaN,8.0,Female,2.0,2.0,NaN,0,0
3,464848,2017-07-02 00:00,1097446,I,359520,13787,3,NaN,3.0,Male,3.0,3.0,2.0,1,0
4,90569,2017-07-02 00:01,663656,C,405490,60305,3,NaN,2.0,Male,2.0,3.0,2.0,1,0


Dataset dimensions: (463291, 15)
Column types:
 session_id                  int64
DateTime                      str
user_id                     int64
product                       str
campaign_id                 int64
webpage_id                  int64
product_category_1          int64
product_category_2        float64
user_group_id             float64
gender                        str
age_level                 float64
user_depth                float64
city_development_index    float64
var_1                       int64
is_click                    int64
dtype: object
Target variable (CTR) distribution:
 is_click
0    431960
1     31331
Name: count, dtype: int64
Missing values:
 session_id                     0
DateTime                       0
user_id                        0
product                        0
campaign_id                    0
webpage_id                     0
product_category_1             0
product_category_2        365854
user_group_id              18243
gender            

Step 2: Exploratory Data Analysis (EDA)

In [8]:
# Target Distribution

## What percentage of ads get clicked?
display(df['is_click'].value_counts(normalize=True))

## Is the dataset severely imbalanced?
clicked_percentage = round(df['is_click'].mean() * 100, 2)
print("Percentage of ads clicked:", clicked_percentage, "%")
print("Percentage of ads not clicked:", round(100 - clicked_percentage, 2), "%")
print("Is the dataset severely imbalanced?", "Yes" if clicked_percentage < 5 else "No")

## Do you need resampling techniques?

is_click
0    0.932373
1    0.067627
Name: proportion, dtype: float64

Percentage of ads clicked: 6.76 %
Percentage of ads not clicked: 93.24 %
Is the dataset severely imbalanced? No


In [19]:
#Temporal Patterns

## Create a new column for the hour of the day from the timestamp
df['hour'] = pd.to_datetime(df['DateTime']).dt.hour
df['month'] = pd.to_datetime(df['DateTime']).dt.month
df['is_weekend'] = (pd.to_datetime(df['DateTime']).dt.dayofweek >= 5).astype(int)
display(df[['DateTime', 'hour', 'month', 'is_weekend']].head())

print("Which hours have highest click rates?")
display(df.groupby('hour')['is_click'].mean().sort_values(ascending=False).head(5))
print("Late night and early morning hours have the highest click rates.")

print("Are weekends different from weekdays?")
display(df.groupby('is_weekend')['is_click'].mean())
print("Click rates are higher on weekends compared to weekdays.")

print("Do certain months perform better?")
print('All months ', df['month'].unique())
display(df.groupby('month')['is_click'].mean().sort_values(ascending=False).head(5))
print("Data is available only for one month.")

,DateTime,hour,month,is_weekend
0,2017-07-02 00:00,0,7,1
1,2017-07-02 00:00,0,7,1
2,2017-07-02 00:00,0,7,1
3,2017-07-02 00:00,0,7,1
4,2017-07-02 00:01,0,7,1


Which hours have highest click rates?


hour
1    0.074608
7    0.073978
6    0.072822
8    0.070271
9    0.070101
Name: is_click, dtype: float64

Late night and early morning hours have the highest click rates.
Are weekends different from weekdays?


is_weekend
0    0.066468
1    0.073262
Name: is_click, dtype: float64

Click rates are higher on weekends compared to weekdays.
Do certain months perform better?
All months  [7]


month
7    0.067627
Name: is_click, dtype: float64

Data is available only for one month.


In [32]:
## User Behavior

### Do certain age groups click more?
print("Which age groups have highest click rates?")
print('=' * 80)
print('All age levels ', df['age_level'].unique())
display(df.groupby('age_level')['is_click'].mean().sort_values(ascending=False).head(5))
print("Youngest & oldest age groups tend to click more on ads.")

### Is there a gender difference in click rates?
print("Click rates by gender:")
print('=' * 80)
display(df.groupby('gender')['is_click'].mean())
print("Click rates are higher for males compared to females.")

### How does user group affect clicking?
print("Which user groups have highest click rates?")
print('=' * 80)
print('All user groups ', df['user_group_id'].unique())
display(df.groupby('user_group_id')['is_click'].mean().sort_values(ascending=False).head(5))
print("Highest & lowest user groups have significantly higher click rates than others.")

Which age groups have highest click rates?
All age levels  [ 4.  2.  3.  1. nan  5.  6.  0.]


age_level
0.0    0.084967
6.0    0.082276
1.0    0.074803
5.0    0.074153
2.0    0.070919
Name: is_click, dtype: float64

Youngest & oldest age groups tend to click more on ads.
Click rates by gender:


gender
Female    0.064445
Male      0.067942
Name: is_click, dtype: float64

Click rates are higher for males compared to females.
Which user groups have highest click rates?
All user groups  [10.  8.  3.  2.  1.  9.  4. nan 11.  7.  5. 12.  6.  0.]


user_group_id
12.0    0.088889
0.0     0.084967
6.0     0.078306
11.0    0.076706
1.0     0.075144
Name: is_click, dtype: float64

Highest & lowest user groups have significantly higher click rates than others.


In [31]:
## Campaign Performance

### Which campaigns have highest CTR?
print("Which campaigns have highest click rates?")
print('Total campaigns ', df['campaign_id'].nunique())
print('Top 5 campaigns:')
print('=' * 80)
display(df.groupby('campaign_id')['is_click'].mean().sort_values(ascending=False).head(5))

### Which products get more clicks?
print("Top 5 products with highest click rates:")
print('=' * 80)
display(df.groupby('product')['is_click'].mean().sort_values(ascending=False).head(5))

### Do certain webpages convert better?
print("Top 5 webpages with highest click rates:")
print('=' * 80)
display(df.groupby('webpage_id')['is_click'].mean().sort_values(ascending=False).head(5))

Which campaigns have highest click rates?
Total campaigns  10
Top 5 campaigns:


campaign_id
405490    0.091307
404347    0.077534
98970     0.076829
396664    0.072624
105960    0.068345
Name: is_click, dtype: float64

Top 5 products with highest click rates:


product
J    0.092700
D    0.071815
H    0.069852
C    0.069149
E    0.068712
Name: is_click, dtype: float64

Top 5 webpages with highest click rates:


webpage_id
60305    0.091307
53587    0.077534
6970     0.076829
51181    0.072624
11085    0.068345
Name: is_click, dtype: float64

Phase 2: Feature Engineering
---

In [33]:
## DateTime Feature Extraction
print("Extracting date and time features from the dataset.")

# - hour: Hour of day (0-23)
# - day_of_week: Day (0=Monday, 6=Sunday)
df['day_of_week'] = pd.to_datetime(df['DateTime']).dt.dayofweek
# - day_of_month: Date of month (1-31)
df['day_of_month'] = pd.to_datetime(df['DateTime']).dt.day
# time_of_day: Categorical (night/morning/afternoon/evening)
df['time_of_day'] = pd.cut(df['hour'], bins=[-1, 5, 11, 17, 23], labels=['night', 'morning', 'afternoon', 'evening'])

display(df.head())

Extracting date and time features from the dataset.


,session_id,DateTime,user_id,product,campaign_id,webpage_id,product_category_1,product_category_2,user_group_id,gender,...,user_depth,city_development_index,var_1,is_click,hour,month,is_weekend,day_of_week,day_of_month,time_of_day
0,140690,2017-07-02 00:00,858557,C,359520,13787,4,NaN,10.0,Female,...,3.0,3.0,0,0,0,7,1,6,2,night
1,333291,2017-07-02 00:00,243253,C,105960,11085,5,NaN,8.0,Female,...,2.0,NaN,0,0,0,7,1,6,2,night
2,129781,2017-07-02 00:00,243253,C,359520,13787,4,NaN,8.0,Female,...,2.0,NaN,0,0,0,7,1,6,2,night
3,464848,2017-07-02 00:00,1097446,I,359520,13787,3,NaN,3.0,Male,...,3.0,2.0,1,0,0,7,1,6,2,night
4,90569,2017-07-02 00:01,663656,C,405490,60305,3,NaN,2.0,Male,...,3.0,2.0,1,0,0,7,1,6,2,night
